In [1]:
!nvidia-smi

Fri May  8 02:44:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/GCP_Assignment_Datasets'   # <-- updated
TRAIN_DIR = os.path.join(BASE, 'train_dataset')
TEST_DIR  = os.path.join(BASE, 'test_dataset')
LABELS_JSON = os.path.join(TRAIN_DIR, 'curated_gcp_marks.json')

WORK = '/content/work'
os.makedirs(WORK, exist_ok=True)
print("Train exists:", os.path.exists(TRAIN_DIR))
print("Test exists:", os.path.exists(TEST_DIR))
print("Labels exists:", os.path.exists(LABELS_JSON))

Mounted at /content/drive
Train exists: True
Test exists: True
Labels exists: False


In [3]:
import os, glob

# List what's actually in the train folder (top level only)
print("Top-level contents of TRAIN_DIR:")
for item in sorted(os.listdir(TRAIN_DIR))[:20]:
    full = os.path.join(TRAIN_DIR, item)
    kind = "DIR " if os.path.isdir(full) else "FILE"
    print(f"  {kind} {item}")

# Search for any JSON file anywhere under BASE
print("\nAll JSON files under BASE:")
for j in glob.glob(os.path.join(BASE, '**/*.json'), recursive=True):
    print(" ", j)


Top-level contents of TRAIN_DIR:
  DIR  231129_CTD
  DIR  Adani GP-III CG
  DIR  Deora Limestone Mine
  DIR  Egypt-New city
  DIR  RDCW-Reddipalayam Limestone Mine
  DIR  Seashell Ras el Hekma
  DIR  UTCL UNCL Additional Area
  DIR  Vedanta GOA Bicholim
  FILE gcp_marks.json
  DIR  scout_966
  DIR  scout_971
  DIR  scout_973

All JSON files under BASE:
  /content/drive/MyDrive/GCP_Assignment_Datasets/train_dataset/gcp_marks.json


In [4]:
LABELS_JSON = os.path.join(TRAIN_DIR, 'gcp_marks.json')
print("Labels exists:", os.path.exists(LABELS_JSON))

Labels exists: True


In [6]:
import json, glob
from collections import Counter
from PIL import Image

with open(LABELS_JSON, 'r') as f:
    labels = json.load(f)

print(f"Total labeled samples: {len(labels)}")

# Class distribution
shapes = [v.get('verified_shape', 'Unknown') for v in labels.values()]
print("Shape distribution:", Counter(shapes))

# Image size sanity check (sample a few)
sample_keys = list(labels.keys())[:5]
for k in sample_keys:
    p = os.path.join(TRAIN_DIR, k)
    if os.path.exists(p):
        im = Image.open(p)
        print(f"{k}: size={im.size}, mark={labels[k]['mark']}, shape={labels[k].get('verified_shape', 'Unknown')}")
    else:
        print(f"MISSING: {p}")

# Check for missing files
missing = [k for k in labels if not os.path.exists(os.path.join(TRAIN_DIR, k))]
print(f"\nMissing image files: {len(missing)}")

# Check for out-of-bounds keypoints
oob = []
for k, v in labels.items():
    x, y = v['mark']['x'], v['mark']['y']
    if not (0 <= x <= 2048 and 0 <= y <= 1365):
        oob.append((k, x, y))
print(f"Out-of-bounds keypoints: {len(oob)}")

# Test set size
test_imgs = glob.glob(os.path.join(TEST_DIR, '**/*.JPG'), recursive=True) + \
            glob.glob(os.path.join(TEST_DIR, '**/*.jpg'), recursive=True)
print(f"\nTest images found: {len(test_imgs)}")

Total labeled samples: 1000
Shape distribution: Counter({'L-Shape': 491, 'Square': 328, 'Cross': 177, 'Unknown': 4})
scout_971/a61f66617a8dcf132dcc2cfa/GCP-11/DJI_20240301143538_0057_V.JPG: size=(4096, 3068), mark={'x': 3272.769145523532, 'y': 1089.3292198344861}, shape=Cross
RDCW-Reddipalayam Limestone Mine/MCDR_ML1_2_3_dataset/LYGCP27/9_DJI_0436.JPG: size=(4096, 2730), mark={'x': 2991.971596265581, 'y': 2445.8607409133274}, shape=Cross
RDCW-Reddipalayam Limestone Mine/MCDR_ML1_2_3_dataset/LYGCP27/9_DJI_0437.JPG: size=(4096, 2730), mark={'x': 2687.926478714031, 'y': 1552.8797519913505}, shape=Cross
RDCW-Reddipalayam Limestone Mine/MCDR_ML1_2_3_dataset/LYGCP27/9_DJI_0438.JPG: size=(4096, 2730), mark={'x': 2862.9958187973157, 'y': 792.5094258695476}, shape=Cross
RDCW-Reddipalayam Limestone Mine/MCDR_ML1_2_3_dataset/LYGCP27/9_DJI_0439.JPG: size=(4096, 2730), mark={'x': 2486.411283510748, 'y': 607.232595526561}, shape=Cross

Missing image files: 0
Out-of-bounds keypoints: 748

Test images

In [8]:
# Re-check OOB using each image's true size
oob_real = []
sample_check = list(labels.items())[:200]  # check 200 to keep it fast
for k, v in sample_check:
    p = os.path.join(TRAIN_DIR, k)
    if not os.path.exists(p): continue
    w, h = Image.open(p).size
    x, y = v['mark']['x'], v['mark']['y']
    if not (0 <= x <= w and 0 <= y <= h):
        oob_real.append((k, x, y, w, h))
print(f"Genuinely out-of-bounds in 200 samples: {len(oob_real)}")
print("Unique shapes in labels:", set(v.get('verified_shape', 'Unknown') for v in labels.values()))

Genuinely out-of-bounds in 200 samples: 0
Unique shapes in labels: {'Cross', 'L-Shape', 'Square', 'Unknown'}


In [9]:
import shutil, time
LOCAL_TRAIN = '/content/train_dataset'
LOCAL_TEST  = '/content/test_dataset'

if not os.path.exists(LOCAL_TRAIN):
    print("Copying train_dataset to local disk...")
    t0 = time.time()
    shutil.copytree(TRAIN_DIR, LOCAL_TRAIN)
    print(f"Done in {time.time()-t0:.1f}s")
else:
    print("Local train already exists")

if not os.path.exists(LOCAL_TEST):
    print("Copying test_dataset to local disk...")
    t0 = time.time()
    shutil.copytree(TEST_DIR, LOCAL_TEST)
    print(f"Done in {time.time()-t0:.1f}s")
else:
    print("Local test already exists")

# Repoint paths to local
TRAIN_DIR = LOCAL_TRAIN
TEST_DIR = LOCAL_TEST
LABELS_JSON = os.path.join(TRAIN_DIR, 'gcp_marks.json')
print("All set. Labels:", os.path.exists(LABELS_JSON))

Copying train_dataset to local disk...
Done in 738.3s
Copying test_dataset to local disk...
Done in 199.6s
All set. Labels: True


In [10]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import numpy as np
import random
from sklearn.model_selection import train_test_split

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

IMG_SIZE = 512
BATCH = 16
EPOCHS = 10
LR = 1e-3
SHAPE_CLASSES = ['Cross', 'Square', 'L-Shape']   # <-- 'L-Shape', not 'L-Shaped'
SHAPE_TO_IDX = {s: i for i, s in enumerate(SHAPE_CLASSES)}
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

Device: cuda


the above is cell 3


In [11]:
class GCPDataset(Dataset):
    def __init__(self, items, root, train=True):
        # items: list of (rel_path, x, y, shape_idx)
        self.items = items
        self.root = root
        self.train = train
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        rel, x, y, cls = self.items[idx]
        path = os.path.join(self.root, rel)
        img = Image.open(path).convert('RGB')
        w, h = img.size  # actual size, varies per image

        img = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)

        if x is not None:
            nx, ny = x / w, y / h
            nx = max(0.0, min(1.0, nx))
            ny = max(0.0, min(1.0, ny))
        else:
            nx, ny = 0.0, 0.0

        arr = np.array(img, dtype=np.float32) / 255.0

        if self.train and random.random() < 0.5:
            arr = arr[:, ::-1, :].copy()
            nx = 1.0 - nx
        if self.train and random.random() < 0.5:
            arr = np.clip(arr * random.uniform(0.85, 1.15), 0, 1)

        tensor = torch.from_numpy(arr).permute(2, 0, 1)
        tensor = self.normalize(tensor)

        return (tensor,
                torch.tensor([nx, ny], dtype=torch.float32),
                torch.tensor(cls if cls is not None else 0, dtype=torch.long),
                rel)

In [13]:
valid_items = []
skipped = 0
for rel, v in labels.items():
    if not os.path.exists(os.path.join(TRAIN_DIR, rel)):
        skipped += 1; continue
    shape = v.get('verified_shape', 'Unknown') # Safely get shape, default to 'Unknown'
    if shape not in SHAPE_TO_IDX:
        skipped += 1; continue   # drops 'Unknown' and any oddballs
    valid_items.append((rel, v['mark']['x'], v['mark']['y'], SHAPE_TO_IDX[shape]))

print(f"Valid samples: {len(valid_items)}  Skipped: {skipped}")

train_items, val_items = train_test_split(
    valid_items, test_size=0.15, random_state=SEED,
    stratify=[it[3] for it in valid_items])

train_ds = GCPDataset(train_items, TRAIN_DIR, train=True)
val_ds   = GCPDataset(val_items, TRAIN_DIR, train=False)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
print(f"Train: {len(train_ds)}  Val: {len(val_ds)}")

Valid samples: 996  Skipped: 4
Train: 846  Val: 150


cell  5


In [14]:
class GCPModel(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        feat_dim = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.kp_head = nn.Sequential(
            nn.Linear(feat_dim, 256), nn.ReLU(inplace=True),
            nn.Linear(256, 2), nn.Sigmoid()  # outputs in [0,1]
        )
        self.cls_head = nn.Sequential(
            nn.Linear(feat_dim, 128), nn.ReLU(inplace=True),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        f = self.backbone(x)
        return self.kp_head(f), self.cls_head(f)

model = GCPModel(num_classes=3).to(device)
print(sum(p.numel() for p in model.parameters())/1e6, "M params")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 84.2MB/s]


11.374405 M params


In [15]:
from collections import Counter

# Class weights from training distribution
counts = Counter(it[3] for it in train_items)
weights = torch.tensor(
    [1.0/counts[i] for i in range(len(SHAPE_CLASSES))],
    dtype=torch.float32, device=device)
weights = weights / weights.sum() * len(SHAPE_CLASSES)
print("Class weights:", weights.cpu().numpy())

# Pre-compute val image sizes once for accurate PCK
print("Caching val image sizes...")
val_sizes = {}
for rel, _, _, _ in val_items:
    val_sizes[rel] = Image.open(os.path.join(TRAIN_DIR, rel)).size

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
mse = nn.MSELoss()
ce = nn.CrossEntropyLoss(weight=weights)

best_val = float('inf')
best_path = os.path.join(WORK, 'best_model.pt')

for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    for imgs, kp, cls, _ in train_loader:
        imgs, kp, cls = imgs.to(device), kp.to(device), cls.to(device)
        pred_kp, pred_cls = model(imgs)
        loss = mse(pred_kp, kp) * 10.0 + ce(pred_cls, cls)
        opt.zero_grad(); loss.backward(); opt.step()
        running += loss.item() * imgs.size(0)
    sched.step()
    train_loss = running / len(train_ds)

    model.eval()
    v_loss = 0.0; v_correct = 0; v_pck10 = 0; v_pck25 = 0; v_pck50 = 0
    with torch.no_grad():
        for imgs, kp, cls, rels in val_loader:
            imgs, kp, cls = imgs.to(device), kp.to(device), cls.to(device)
            pred_kp, pred_cls = model(imgs)
            v_loss += (mse(pred_kp, kp)*10.0 + ce(pred_cls, cls)).item() * imgs.size(0)
            v_correct += (pred_cls.argmax(1) == cls).sum().item()
            for i, r in enumerate(rels):
                w_i, h_i = val_sizes[r]
                dx = (pred_kp[i,0].item() - kp[i,0].item()) * w_i
                dy = (pred_kp[i,1].item() - kp[i,1].item()) * h_i
                d = (dx*dx + dy*dy) ** 0.5
                if d < 10: v_pck10 += 1
                if d < 25: v_pck25 += 1
                if d < 50: v_pck50 += 1
    n = len(val_ds)
    print(f"Epoch {epoch+1}/{EPOCHS}  train_loss={train_loss:.4f}  val_loss={v_loss/n:.4f}  "
          f"acc={v_correct/n:.3f}  PCK@10={v_pck10/n:.3f}  PCK@25={v_pck25/n:.3f}  PCK@50={v_pck50/n:.3f}")

    if v_loss/n < best_val:
        best_val = v_loss/n
        torch.save(model.state_dict(), best_path)
        print("  ✓ saved best")

print("Done.")


Class weights: [1.5811554  0.8500835  0.56876093]
Caching val image sizes...
Epoch 1/10  train_loss=1.3206  val_loss=1.0586  acc=0.847  PCK@10=0.000  PCK@25=0.000  PCK@50=0.000
  ✓ saved best
Epoch 2/10  train_loss=1.0485  val_loss=1.3308  acc=0.807  PCK@10=0.000  PCK@25=0.000  PCK@50=0.000
Epoch 3/10  train_loss=0.9976  val_loss=0.8471  acc=0.940  PCK@10=0.000  PCK@25=0.000  PCK@50=0.000
  ✓ saved best
Epoch 4/10  train_loss=0.9647  val_loss=0.9345  acc=0.887  PCK@10=0.000  PCK@25=0.000  PCK@50=0.000
Epoch 5/10  train_loss=0.8435  val_loss=0.8579  acc=0.960  PCK@10=0.000  PCK@25=0.000  PCK@50=0.000
Epoch 6/10  train_loss=0.7797  val_loss=0.7257  acc=0.993  PCK@10=0.000  PCK@25=0.000  PCK@50=0.000
  ✓ saved best
Epoch 7/10  train_loss=0.7499  val_loss=0.7047  acc=1.000  PCK@10=0.000  PCK@25=0.000  PCK@50=0.000
  ✓ saved best
Epoch 8/10  train_loss=0.7199  val_loss=0.6974  acc=1.000  PCK@10=0.000  PCK@25=0.000  PCK@50=0.000
  ✓ saved best
Epoch 9/10  train_loss=0.7336  val_loss=0.6991  

In [16]:
# Sanity check the keypoint predictions
model.eval()
with torch.no_grad():
    sample_batch = next(iter(val_loader))
    imgs, kp, cls, rels = sample_batch
    imgs = imgs.to(device)
    pred_kp, pred_cls = model(imgs)
    print("First 5 predicted keypoints (normalized):")
    print(pred_kp[:5].cpu().numpy())
    print("First 5 true keypoints (normalized):")
    print(kp[:5].numpy())
    print("\nFirst 5 rels:", rels[:5])
    print("First 5 in val_sizes?", [r in val_sizes for r in rels[:5]])

First 5 predicted keypoints (normalized):
[[0.5028346  0.51195663]
 [0.49819446 0.50837487]
 [0.5051607  0.5091    ]
 [0.49850366 0.50563425]
 [0.4958608  0.5020551 ]]
First 5 true keypoints (normalized):
[[0.06874042 0.0791025 ]
 [0.5999625  0.38202712]
 [0.7275257  0.79047364]
 [0.4424419  0.5000709 ]
 [0.37639165 0.40424094]]

First 5 rels: ('Seashell Ras el Hekma/Survey 3/GCP21/DJI_20240605114057_0054.JPG', 'UTCL UNCL Additional Area/Survey-1/GCP-65/DJI_20240424164037_0042_V.JPG', '231129_CTD/231129_CTD_GDA94/GCP9/DJI_20231129125239_0192.JPG', 'Deora Limestone Mine/MCDR_2024/GCP27/DJI_20240420125129_0259.JPG', 'Egypt-New city/Survey 1/18G70/DJI_20241107093748_0015.JPG')
First 5 in val_sizes? [True, True, True, True, True]


In [17]:
# === RETRAIN with fixed loss balance ===
# Re-init the model from scratch (clears the collapsed weights)
model = GCPModel(num_classes=3).to(device)

EPOCHS_RETRY = 8
LR_RETRY = 5e-4

opt = torch.optim.AdamW(model.parameters(), lr=LR_RETRY, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS_RETRY)

# Smooth L1 is much more stable than MSE for keypoint regression
kp_loss_fn = nn.SmoothL1Loss(beta=0.05)
ce = nn.CrossEntropyLoss(weight=weights)

KP_WEIGHT = 100.0   # was 10 — way too low

best_val = float('inf')
best_path = os.path.join(WORK, 'best_model.pt')

for epoch in range(EPOCHS_RETRY):
    model.train()
    running_total, running_kp, running_cls = 0.0, 0.0, 0.0
    for imgs, kp, cls, _ in train_loader:
        imgs, kp, cls = imgs.to(device), kp.to(device), cls.to(device)
        pred_kp, pred_cls = model(imgs)
        l_kp = kp_loss_fn(pred_kp, kp) * KP_WEIGHT
        l_cls = ce(pred_cls, cls)
        loss = l_kp + l_cls
        opt.zero_grad(); loss.backward(); opt.step()
        running_total += loss.item() * imgs.size(0)
        running_kp += l_kp.item() * imgs.size(0)
        running_cls += l_cls.item() * imgs.size(0)
    sched.step()
    n_tr = len(train_ds)

    model.eval()
    v_loss = 0.0; v_correct = 0; v_pck10 = 0; v_pck25 = 0; v_pck50 = 0
    with torch.no_grad():
        for imgs, kp, cls, rels in val_loader:
            imgs, kp, cls = imgs.to(device), kp.to(device), cls.to(device)
            pred_kp, pred_cls = model(imgs)
            v_loss += (kp_loss_fn(pred_kp, kp)*KP_WEIGHT + ce(pred_cls, cls)).item() * imgs.size(0)
            v_correct += (pred_cls.argmax(1) == cls).sum().item()
            for i, r in enumerate(rels):
                w_i, h_i = val_sizes[r]
                dx = (pred_kp[i,0].item() - kp[i,0].item()) * w_i
                dy = (pred_kp[i,1].item() - kp[i,1].item()) * h_i
                d = (dx*dx + dy*dy) ** 0.5
                if d < 10: v_pck10 += 1
                if d < 25: v_pck25 += 1
                if d < 50: v_pck50 += 1
    n = len(val_ds)
    print(f"Epoch {epoch+1}/{EPOCHS_RETRY}  train_total={running_total/n_tr:.3f}  "
          f"train_kp={running_kp/n_tr:.3f}  train_cls={running_cls/n_tr:.3f}  "
          f"val_loss={v_loss/n:.3f}  acc={v_correct/n:.3f}  "
          f"PCK@10={v_pck10/n:.3f}  PCK@25={v_pck25/n:.3f}  PCK@50={v_pck50/n:.3f}")

    if v_loss/n < best_val:
        best_val = v_loss/n
        torch.save(model.state_dict(), best_path)
        print("  ✓ saved best")

print("Retraining done.")

Epoch 1/8  train_total=21.491  train_kp=20.571  train_cls=0.919  val_loss=21.867  acc=0.547  PCK@10=0.000  PCK@25=0.000  PCK@50=0.000
  ✓ saved best
Epoch 2/8  train_total=20.007  train_kp=19.509  train_cls=0.498  val_loss=22.136  acc=0.893  PCK@10=0.000  PCK@25=0.000  PCK@50=0.000
Epoch 3/8  train_total=20.268  train_kp=19.857  train_cls=0.411  val_loss=20.707  acc=0.953  PCK@10=0.000  PCK@25=0.000  PCK@50=0.000
  ✓ saved best


KeyboardInterrupt: 

In [18]:
# === FINAL ATTEMPT: clean rebuild + train + inference in one shot ===
import torch, torch.nn as nn, json, glob, shutil
from torchvision import models, transforms
import numpy as np
from PIL import Image

# --- Model: no sigmoid, dropout, smaller heads ---
class GCPModelFinal(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        bb = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        d = bb.fc.in_features
        bb.fc = nn.Identity()
        self.bb = bb
        self.kp = nn.Linear(d, 2)
        self.cls = nn.Linear(d, num_classes)
        # Init kp head small so it doesn't blow up
        nn.init.normal_(self.kp.weight, std=0.01)
        nn.init.constant_(self.kp.bias, 0.5)

    def forward(self, x):
        f = self.bb(x)
        return self.kp(f), self.cls(f)

model = GCPModelFinal(num_classes=3).to(device)

EPOCHS_F = 6
opt = torch.optim.AdamW([
    {'params': model.bb.parameters(), 'lr': 5e-4},
    {'params': model.kp.parameters(), 'lr': 2e-3},
    {'params': model.cls.parameters(), 'lr': 2e-3},
], weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS_F)
mse = nn.MSELoss()
ce = nn.CrossEntropyLoss(weight=weights)

best_val = float('inf')
best_path = os.path.join(WORK, 'best_model.pt')

for epoch in range(EPOCHS_F):
    model.train()
    rt = 0.0
    for imgs, kp, cls, _ in train_loader:
        imgs, kp, cls = imgs.to(device), kp.to(device), cls.to(device)
        pred_kp, pred_cls = model(imgs)
        # Direct MSE, weighted very high; CE small
        loss = mse(pred_kp, kp) * 200.0 + ce(pred_cls, cls) * 0.5
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        rt += loss.item() * imgs.size(0)
    sched.step()

    model.eval()
    v_loss = 0.0; v_correct = 0; v_p10 = 0; v_p25 = 0; v_p50 = 0
    with torch.no_grad():
        for imgs, kp, cls, rels in val_loader:
            imgs, kp, cls = imgs.to(device), kp.to(device), cls.to(device)
            pk, pc = model(imgs)
            v_loss += (mse(pk, kp)*200.0 + ce(pc, cls)*0.5).item() * imgs.size(0)
            v_correct += (pc.argmax(1) == cls).sum().item()
            pk_c = pk.clamp(0, 1)
            for i, r in enumerate(rels):
                w_i, h_i = val_sizes[r]
                dx = (pk_c[i,0].item() - kp[i,0].item()) * w_i
                dy = (pk_c[i,1].item() - kp[i,1].item()) * h_i
                d = (dx*dx + dy*dy) ** 0.5
                if d < 10: v_p10 += 1
                if d < 25: v_p25 += 1
                if d < 50: v_p50 += 1
    n = len(val_ds)
    print(f"Ep {epoch+1}/{EPOCHS_F} train={rt/len(train_ds):.3f} val={v_loss/n:.3f} "
          f"acc={v_correct/n:.3f} PCK10={v_p10/n:.3f} PCK25={v_p25/n:.3f} PCK50={v_p50/n:.3f}")
    if v_loss/n < best_val:
        best_val = v_loss/n
        torch.save(model.state_dict(), best_path)
        print("  ✓ saved")

print("\n=== TRAINING DONE — RUNNING INFERENCE ===\n")

# --- Inference ---
model.load_state_dict(torch.load(best_path, map_location=device))
model.eval()

test_paths = []
for ext in ('*.JPG','*.jpg','*.jpeg','*.JPEG','*.png','*.PNG'):
    test_paths += glob.glob(os.path.join(TEST_DIR, '**', ext), recursive=True)
print(f"Test images: {len(test_paths)}")

normalize = transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
predictions = {}
buf_imgs, buf_rels, buf_sizes = [], [], []

def flush():
    if not buf_imgs: return
    x = torch.stack(buf_imgs).to(device)
    with torch.no_grad():
        kp, cls = model(x)
    kp = kp.clamp(0, 1).cpu().numpy()
    ci = cls.argmax(1).cpu().numpy()
    for i, rel in enumerate(buf_rels):
        w_i, h_i = buf_sizes[i]
        predictions[rel] = {
            "mark": {"x": round(float(kp[i,0])*w_i, 2),
                     "y": round(float(kp[i,1])*h_i, 2)},
            "verified_shape": SHAPE_CLASSES[int(ci[i])]
        }

for p in test_paths:
    rel = os.path.relpath(p, TEST_DIR).replace('\\','/')
    img = Image.open(p).convert('RGB')
    buf_sizes.append(img.size)
    arr = np.array(img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR), dtype=np.float32)/255.0
    t = normalize(torch.from_numpy(arr).permute(2,0,1))
    buf_imgs.append(t); buf_rels.append(rel)
    if len(buf_imgs) >= BATCH:
        flush(); buf_imgs, buf_rels, buf_sizes = [], [], []
flush()

out_path = os.path.join(WORK, 'predictions.json')
with open(out_path, 'w') as f:
    json.dump(predictions, f, indent=2)
shutil.copy(out_path, os.path.join(BASE, 'predictions.json'))
shutil.copy(best_path, os.path.join(BASE, 'best_model.pt'))
print(f"\n✓ Wrote {len(predictions)} predictions")
print(f"✓ predictions.json and best_model.pt copied to {BASE}")
print("\nSample predictions:")
import itertools
for k, v in itertools.islice(predictions.items(), 3):
    print(" ", k, "->", v)


Ep 1/6 train=24.965 val=16.418 acc=0.373 PCK10=0.000 PCK25=0.000 PCK50=0.000
  ✓ saved
Ep 2/6 train=17.385 val=18.501 acc=0.493 PCK10=0.000 PCK25=0.000 PCK50=0.000


KeyboardInterrupt: 

In [19]:
# Hard diagnostic — what is the model actually outputting?
model.eval()
with torch.no_grad():
    imgs, kp, cls, rels = next(iter(val_loader))
    imgs = imgs.to(device)
    pk, pc = model(imgs)
    print("Predictions (raw, first 5):")
    print(pk[:5].cpu().numpy())
    print("\nGround truth (first 5):")
    print(kp[:5].numpy())
    print("\nPrediction stats: min={:.3f} max={:.3f} mean={:.3f} std={:.4f}".format(
        pk.min().item(), pk.max().item(), pk.mean().item(), pk.std().item()))
    print("GT stats:        min={:.3f} max={:.3f} mean={:.3f} std={:.4f}".format(
        kp.min().item(), kp.max().item(), kp.mean().item(), kp.std().item()))

# Also check: does train data even have varied keypoints?
print("\nTraining keypoint spread (first 10 samples):")
for it in train_items[:10]:
    rel, x, y, c = it
    img = Image.open(os.path.join(TRAIN_DIR, rel))
    w, h = img.size
    print(f"  {x/w:.3f}, {y/h:.3f}  (image {w}x{h})")


Predictions (raw, first 5):
[[0.59474355 0.48273686]
 [0.59115416 0.50512373]
 [0.59729266 0.48190856]
 [0.61587083 0.4960075 ]
 [0.64314705 0.51969993]]

Ground truth (first 5):
[[0.06874042 0.0791025 ]
 [0.5999625  0.38202712]
 [0.7275257  0.79047364]
 [0.4424419  0.5000709 ]
 [0.37639165 0.40424094]]

Prediction stats: min=0.480 max=0.643 mean=0.557 std=0.0557
GT stats:        min=0.069 max=0.916 mean=0.464 std=0.2277

Training keypoint spread (first 10 samples):
  0.390, 0.735  (image 4096x2730)
  0.410, 0.070  (image 4096x3068)
  0.133, 0.249  (image 4096x2730)
  0.146, 0.696  (image 4096x3068)
  0.576, 0.423  (image 4096x3068)
  0.061, 0.884  (image 4096x3068)
  0.384, 0.825  (image 4096x3068)
  0.625, 0.678  (image 4096x2730)
  0.096, 0.890  (image 4096x2730)
  0.491, 0.730  (image 4096x2730)


In [20]:
# === LAST ATTEMPT: head-focused training with L1 loss ===
# Re-init fresh model
class GCPModelF2(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        bb = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        d = bb.fc.in_features
        bb.fc = nn.Identity()
        self.bb = bb
        self.kp = nn.Sequential(
            nn.Linear(d, 512), nn.ReLU(inplace=True),
            nn.Linear(512, 2)
        )
        self.cls = nn.Linear(d, num_classes)
        # Init final kp layer near zero so output starts at bias=0.5
        nn.init.normal_(self.kp[-1].weight, std=0.001)
        nn.init.constant_(self.kp[-1].bias, 0.5)

    def forward(self, x):
        f = self.bb(x)
        return self.kp(f), self.cls(f)

model = GCPModelF2(num_classes=3).to(device)

EPOCHS_F = 8
opt = torch.optim.AdamW([
    {'params': model.bb.parameters(),  'lr': 1e-4},   # backbone slow
    {'params': model.kp.parameters(),  'lr': 5e-3},   # kp head FAST
    {'params': model.cls.parameters(), 'lr': 1e-3},
], weight_decay=1e-5)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS_F)

# L1 — gives constant gradient magnitude regardless of error size,
# which prevents the "settle near mean" failure mode of MSE
l1 = nn.L1Loss()
ce = nn.CrossEntropyLoss(weight=weights)

best_val = float('inf')
best_path = os.path.join(WORK, 'best_model.pt')

for epoch in range(EPOCHS_F):
    model.train()
    rt = 0.0
    for imgs, kp, cls, _ in train_loader:
        imgs, kp, cls = imgs.to(device), kp.to(device), cls.to(device)
        pred_kp, pred_cls = model(imgs)
        loss = l1(pred_kp, kp) * 50.0 + ce(pred_cls, cls)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        rt += loss.item() * imgs.size(0)
    sched.step()

    model.eval()
    v_loss = 0.0; v_correct = 0; v_p10 = 0; v_p25 = 0; v_p50 = 0
    pred_std_sum = 0.0
    n_batches = 0
    with torch.no_grad():
        for imgs, kp, cls, rels in val_loader:
            imgs, kp, cls = imgs.to(device), kp.to(device), cls.to(device)
            pk, pc = model(imgs)
            v_loss += (l1(pk, kp)*50.0 + ce(pc, cls)).item() * imgs.size(0)
            v_correct += (pc.argmax(1) == cls).sum().item()
            pred_std_sum += pk.std().item(); n_batches += 1
            pk_c = pk.clamp(0, 1)
            for i, r in enumerate(rels):
                w_i, h_i = val_sizes[r]
                dx = (pk_c[i,0].item() - kp[i,0].item()) * w_i
                dy = (pk_c[i,1].item() - kp[i,1].item()) * h_i
                d = (dx*dx + dy*dy) ** 0.5
                if d < 10: v_p10 += 1
                if d < 25: v_p25 += 1
                if d < 50: v_p50 += 1
    n = len(val_ds)
    print(f"Ep {epoch+1}/{EPOCHS_F} train={rt/len(train_ds):.3f} val={v_loss/n:.3f} "
          f"acc={v_correct/n:.3f} pred_std={pred_std_sum/n_batches:.3f} "
          f"PCK10={v_p10/n:.3f} PCK25={v_p25/n:.3f} PCK50={v_p50/n:.3f}")
    if v_loss/n < best_val:
        best_val = v_loss/n
        torch.save(model.state_dict(), best_path)
        print("  ✓ saved")

print("\n=== INFERENCE ===")
model.load_state_dict(torch.load(best_path, map_location=device))
model.eval()

test_paths = []
for ext in ('*.JPG','*.jpg','*.jpeg','*.JPEG','*.png','*.PNG'):
    test_paths += glob.glob(os.path.join(TEST_DIR, '**', ext), recursive=True)

normalize = transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
predictions = {}
buf_imgs, buf_rels, buf_sizes = [], [], []

def flush():
    if not buf_imgs: return
    x = torch.stack(buf_imgs).to(device)
    with torch.no_grad():
        kp, cls = model(x)
    kp = kp.clamp(0, 1).cpu().numpy()
    ci = cls.argmax(1).cpu().numpy()
    for i, rel in enumerate(buf_rels):
        w_i, h_i = buf_sizes[i]
        predictions[rel] = {
            "mark": {"x": round(float(kp[i,0])*w_i, 2),
                     "y": round(float(kp[i,1])*h_i, 2)},
            "verified_shape": SHAPE_CLASSES[int(ci[i])]
        }

for p in test_paths:
    rel = os.path.relpath(p, TEST_DIR).replace('\\','/')
    img = Image.open(p).convert('RGB')
    buf_sizes.append(img.size)
    arr = np.array(img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR), dtype=np.float32)/255.0
    t = normalize(torch.from_numpy(arr).permute(2,0,1))
    buf_imgs.append(t); buf_rels.append(rel)
    if len(buf_imgs) >= BATCH:
        flush(); buf_imgs, buf_rels, buf_sizes = [], [], []
flush()

out_path = os.path.join(WORK, 'predictions.json')
with open(out_path, 'w') as f:
    json.dump(predictions, f, indent=2)
shutil.copy(out_path, os.path.join(BASE, 'predictions.json'))
shutil.copy(best_path, os.path.join(BASE, 'best_model.pt'))
print(f"✓ Wrote {len(predictions)} predictions to Drive")

Ep 1/8 train=14.596 val=11.618 acc=0.953 pred_std=0.035 PCK10=0.000 PCK25=0.000 PCK50=0.000
  ✓ saved
Ep 2/8 train=11.230 val=11.424 acc=0.993 pred_std=0.008 PCK10=0.000 PCK25=0.000 PCK50=0.000
  ✓ saved


KeyboardInterrupt: 

In [21]:
# === STAGE 1: Heatmap-based dataset ===
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import numpy as np
import random

HEATMAP_SIZE = 128   # output heatmap is 128x128 (input 512 → /4 downsample)
SIGMA = 2.0          # Gaussian peak width in heatmap pixels

def make_heatmap(x_norm, y_norm, size=HEATMAP_SIZE, sigma=SIGMA):
    """Generate a 2D Gaussian heatmap with peak at (x_norm, y_norm)."""
    cx = x_norm * (size - 1)
    cy = y_norm * (size - 1)
    xs = np.arange(size, dtype=np.float32)
    ys = np.arange(size, dtype=np.float32)
    xx, yy = np.meshgrid(xs, ys)
    hm = np.exp(-((xx - cx)**2 + (yy - cy)**2) / (2 * sigma**2))
    return hm.astype(np.float32)

class GCPDatasetHM(Dataset):
    def __init__(self, items, root, train=True):
        self.items = items
        self.root = root
        self.train = train
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        rel, x, y, cls = self.items[idx]
        path = os.path.join(self.root, rel)
        img = Image.open(path).convert('RGB')
        w, h = img.size

        img = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)

        nx, ny = x / w, y / h
        nx = max(0.0, min(1.0, nx))
        ny = max(0.0, min(1.0, ny))

        arr = np.array(img, dtype=np.float32) / 255.0

        if self.train and random.random() < 0.5:
            arr = arr[:, ::-1, :].copy()
            nx = 1.0 - nx
        if self.train and random.random() < 0.5:
            arr = np.clip(arr * random.uniform(0.85, 1.15), 0, 1)

        hm = make_heatmap(nx, ny)

        tensor = torch.from_numpy(arr).permute(2, 0, 1)
        tensor = self.normalize(tensor)

        return (tensor,
                torch.from_numpy(hm),
                torch.tensor(cls, dtype=torch.long),
                torch.tensor([nx, ny], dtype=torch.float32),
                rel)

# Quick visual sanity check
sample_ds = GCPDatasetHM(train_items, TRAIN_DIR, train=False)
img_t, hm_t, cls_t, kp_t, rel_t = sample_ds[0]
print(f"Image tensor: {img_t.shape}")
print(f"Heatmap: {hm_t.shape}, peak value: {hm_t.max():.3f}, peak loc: {np.unravel_index(hm_t.argmax(), hm_t.shape)}")
print(f"True normalized kp: ({kp_t[0]:.3f}, {kp_t[1]:.3f})")
print(f"Expected peak at heatmap coords: ({kp_t[0]*127:.1f}, {kp_t[1]*127:.1f}) — y, x in numpy is ({kp_t[1]*127:.1f}, {kp_t[0]*127:.1f})")

Image tensor: torch.Size([3, 512, 512])
Heatmap: torch.Size([128, 128]), peak value: 0.959, peak loc: (np.int64(93), np.int64(50))
True normalized kp: (0.390, 0.735)
Expected peak at heatmap coords: (49.5, 93.4) — y, x in numpy is (93.4, 49.5)


/tmp/ipykernel_3760/1520992013.py:69: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
  print(f"Heatmap: {hm_t.shape}, peak value: {hm_t.max():.3f}, peak loc: {np.unravel_index(hm_t.argmax(), hm_t.shape)}")


In [22]:
# === STAGE 2: Heatmap model with UNet-style decoder ===
import torch.nn.functional as F

class HeatmapModel(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        bb = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        # ResNet18 stem: conv1+bn+relu+maxpool → 1/4 spatial
        # layer1 → 1/4, layer2 → 1/8, layer3 → 1/16, layer4 → 1/32
        self.stem = nn.Sequential(bb.conv1, bb.bn1, bb.relu, bb.maxpool)
        self.layer1 = bb.layer1   # out: 64 ch, 1/4
        self.layer2 = bb.layer2   # out: 128 ch, 1/8
        self.layer3 = bb.layer3   # out: 256 ch, 1/16
        self.layer4 = bb.layer4   # out: 512 ch, 1/32

        # Decoder: upsample 1/32 → 1/16 → 1/8 → 1/4
        # With skip connections from encoder
        self.up3 = nn.Sequential(
            nn.Conv2d(512 + 256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True))
        self.up2 = nn.Sequential(
            nn.Conv2d(256 + 128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True))
        self.up1 = nn.Sequential(
            nn.Conv2d(128 + 64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True))
        # Final 1x1 conv to produce a single-channel heatmap
        self.heatmap_head = nn.Conv2d(64, 1, 1)

        # Classification head: GAP on deepest features
        self.cls_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x0 = self.stem(x)        # [B, 64, 128, 128]   1/4
        x1 = self.layer1(x0)     # [B, 64, 128, 128]   1/4
        x2 = self.layer2(x1)     # [B, 128, 64, 64]    1/8
        x3 = self.layer3(x2)     # [B, 256, 32, 32]    1/16
        x4 = self.layer4(x3)     # [B, 512, 16, 16]    1/32

        # Decoder with skip connections
        u3 = F.interpolate(x4, scale_factor=2, mode='bilinear', align_corners=False)
        u3 = self.up3(torch.cat([u3, x3], dim=1))   # [B, 256, 32, 32]
        u2 = F.interpolate(u3, scale_factor=2, mode='bilinear', align_corners=False)
        u2 = self.up2(torch.cat([u2, x2], dim=1))   # [B, 128, 64, 64]
        u1 = F.interpolate(u2, scale_factor=2, mode='bilinear', align_corners=False)
        u1 = self.up1(torch.cat([u1, x1], dim=1))   # [B, 64, 128, 128]

        heatmap = self.heatmap_head(u1).squeeze(1)  # [B, 128, 128]
        cls_logits = self.cls_head(x4)              # [B, 3]
        return heatmap, cls_logits

# Sanity test
hm_model = HeatmapModel(num_classes=3).to(device)
test_input = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(device)
with torch.no_grad():
    hm_out, cls_out = hm_model(test_input)
print(f"Heatmap output shape: {hm_out.shape}  (expected [2, 128, 128])")
print(f"Class output shape:   {cls_out.shape}  (expected [2, 3])")
print(f"Total params: {sum(p.numel() for p in hm_model.parameters())/1e6:.2f}M")

Heatmap output shape: torch.Size([2, 128, 128])  (expected [2, 128, 128])
Class output shape:   torch.Size([2, 3])  (expected [2, 3])
Total params: 13.50M


In [23]:
# === STAGE 3: Train heatmap model ===

# Build dataloaders with new dataset class
train_ds_hm = GCPDatasetHM(train_items, TRAIN_DIR, train=True)
val_ds_hm   = GCPDatasetHM(val_items, TRAIN_DIR, train=False)
train_loader_hm = DataLoader(train_ds_hm, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)
val_loader_hm   = DataLoader(val_ds_hm, batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

EPOCHS_HM = 12
opt = torch.optim.AdamW([
    {'params': list(hm_model.stem.parameters()) +
               list(hm_model.layer1.parameters()) +
               list(hm_model.layer2.parameters()) +
               list(hm_model.layer3.parameters()) +
               list(hm_model.layer4.parameters()), 'lr': 1e-4},
    {'params': list(hm_model.up3.parameters()) +
               list(hm_model.up2.parameters()) +
               list(hm_model.up1.parameters()) +
               list(hm_model.heatmap_head.parameters()) +
               list(hm_model.cls_head.parameters()), 'lr': 1e-3},
], weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS_HM)

mse = nn.MSELoss()
ce = nn.CrossEntropyLoss(weight=weights)
HM_W = 1000.0   # heatmap MSE values are tiny (Gaussian peak ~1, rest ~0)

def heatmap_to_xy(hm):
    """hm: [B, H, W] tensor. Returns (B, 2) normalized (x, y)."""
    B, H, W = hm.shape
    flat = hm.view(B, -1)
    idx = flat.argmax(dim=1)
    ys = (idx // W).float() / (H - 1)
    xs = (idx %  W).float() / (W - 1)
    return torch.stack([xs, ys], dim=1)

best_val = float('inf')
best_path = os.path.join(WORK, 'best_model.pt')

for epoch in range(EPOCHS_HM):
    hm_model.train()
    rt = 0.0
    for imgs, hms, cls, _, _ in train_loader_hm:
        imgs, hms, cls = imgs.to(device), hms.to(device), cls.to(device)
        pred_hm, pred_cls = hm_model(imgs)
        loss = mse(pred_hm, hms) * HM_W + ce(pred_cls, cls)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(hm_model.parameters(), 1.0)
        opt.step()
        rt += loss.item() * imgs.size(0)
    sched.step()

    hm_model.eval()
    v_loss = 0.0; v_correct = 0; v_p10 = 0; v_p25 = 0; v_p50 = 0
    with torch.no_grad():
        for imgs, hms, cls, kps, rels in val_loader_hm:
            imgs, hms, cls, kps = imgs.to(device), hms.to(device), cls.to(device), kps.to(device)
            pred_hm, pred_cls = hm_model(imgs)
            v_loss += (mse(pred_hm, hms)*HM_W + ce(pred_cls, cls)).item() * imgs.size(0)
            v_correct += (pred_cls.argmax(1) == cls).sum().item()
            pred_xy = heatmap_to_xy(pred_hm)
            for i, r in enumerate(rels):
                w_i, h_i = val_sizes[r]
                dx = (pred_xy[i,0].item() - kps[i,0].item()) * w_i
                dy = (pred_xy[i,1].item() - kps[i,1].item()) * h_i
                d = (dx*dx + dy*dy) ** 0.5
                if d < 10: v_p10 += 1
                if d < 25: v_p25 += 1
                if d < 50: v_p50 += 1
    n = len(val_ds_hm)
    print(f"Ep {epoch+1}/{EPOCHS_HM} train={rt/len(train_ds_hm):.4f} val={v_loss/n:.4f} "
          f"acc={v_correct/n:.3f} PCK10={v_p10/n:.3f} PCK25={v_p25/n:.3f} PCK50={v_p50/n:.3f}")

    if v_loss/n < best_val:
        best_val = v_loss/n
        torch.save(hm_model.state_dict(), best_path)
        print("  ✓ saved")

print("Training done.")

Ep 1/12 train=24.3222 val=0.9150 acc=0.993 PCK10=0.047 PCK25=0.100 PCK50=0.113
  ✓ saved
Ep 2/12 train=1.0069 val=0.9929 acc=0.993 PCK10=0.040 PCK25=0.080 PCK50=0.120
Ep 3/12 train=0.8743 val=1.1537 acc=1.000 PCK10=0.007 PCK25=0.027 PCK50=0.193
Ep 4/12 train=0.8848 val=0.8746 acc=1.000 PCK10=0.080 PCK25=0.287 PCK50=0.387
  ✓ saved
Ep 5/12 train=0.8660 val=0.8452 acc=1.000 PCK10=0.127 PCK25=0.307 PCK50=0.393
  ✓ saved
Ep 6/12 train=0.8611 val=0.8986 acc=1.000 PCK10=0.187 PCK25=0.413 PCK50=0.460
Ep 7/12 train=0.7079 val=0.6718 acc=1.000 PCK10=0.193 PCK25=0.427 PCK50=0.513
  ✓ saved
Ep 8/12 train=0.6026 val=0.5567 acc=1.000 PCK10=0.200 PCK25=0.467 PCK50=0.567
  ✓ saved
Ep 9/12 train=0.5272 val=0.4786 acc=1.000 PCK10=0.207 PCK25=0.513 PCK50=0.627
  ✓ saved
Ep 10/12 train=0.4383 val=0.5083 acc=1.000 PCK10=0.227 PCK25=0.553 PCK50=0.653
Ep 11/12 train=0.3840 val=0.4037 acc=1.000 PCK10=0.247 PCK25=0.573 PCK50=0.707
  ✓ saved
Ep 12/12 train=0.3585 val=0.3888 acc=1.000 PCK10=0.247 PCK25=0.587 PC

In [25]:
# === STAGE 4: Inference with soft-argmax for sub-pixel precision ===
import torch.nn.functional as F

hm_model.load_state_dict(torch.load(best_path, map_location=device))
hm_model.eval()

def soft_argmax_2d(hm, beta=100.0):
    """
    hm: [B, H, W] heatmap.
    Returns normalized (x, y) in [0, 1] using softmax-weighted spatial average.
    Sub-pixel accurate, much better than discrete argmax for PCK@10.
    """
    B, H, W = hm.shape
    flat = hm.view(B, -1) * beta
    soft = F.softmax(flat, dim=1).view(B, H, W)
    ys = torch.arange(H, device=hm.device, dtype=torch.float32).view(1, H, 1)
    xs = torch.arange(W, device=hm.device, dtype=torch.float32).view(1, 1, W)
    cy = (soft * ys).sum(dim=(1, 2)) / (H - 1)
    cx = (soft * xs).sum(dim=(1, 2)) / (W - 1)
    return torch.stack([cx, cy], dim=1)

test_paths = []
for ext in ('*.JPG','*.jpg','*.jpeg','*.JPEG','*.png','*.PNG'):
    test_paths += glob.glob(os.path.join(TEST_DIR, '**', ext), recursive=True)
print(f"Test images: {len(test_paths)}")

normalize = transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
predictions = {}
buf_imgs, buf_rels, buf_sizes = [], [], []

def flush_hm():
    if not buf_imgs: return
    x = torch.stack(buf_imgs).to(device)
    with torch.no_grad():
        pred_hm, pred_cls = hm_model(x)
        pred_xy = soft_argmax_2d(pred_hm).cpu().numpy()
        ci = pred_cls.argmax(1).cpu().numpy()
    for i, rel in enumerate(buf_rels):
        w_i, h_i = buf_sizes[i]
        predictions[rel] = {
            "mark": {"x": round(float(pred_xy[i,0]) * w_i, 2),
                     "y": round(float(pred_xy[i,1]) * h_i, 2)},
            "verified_shape": SHAPE_CLASSES[int(ci[i])]
        }

for p in test_paths:
    rel = os.path.relpath(p, TEST_DIR).replace('\\','/')
    img = Image.open(p).convert('RGB')
    buf_sizes.append(img.size)
    arr = np.array(img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR), dtype=np.float32)/255.0
    t = normalize(torch.from_numpy(arr).permute(2,0,1))
    buf_imgs.append(t); buf_rels.append(rel)
    if len(buf_imgs) >= BATCH:
        flush_hm(); buf_imgs, buf_rels, buf_sizes = [], [], []
flush_hm()

out_path = os.path.join(WORK, 'predictions.json')
with open(out_path, 'w') as f:
    json.dump(predictions, f, indent=2)
# The predictions.json and best_model.pt are already saved to the WORK directory.
# Removed shutil.copy to Google Drive to avoid Read-only file system error.
print(f"\n✓ Wrote {len(predictions)} predictions to {out_path}")
print(f"✓ Best model saved to {best_path}")
print("\nSample predictions:")
import itertools
for k, v in itertools.islice(predictions.items(), 3):
    print(" ", k, "->", v)


Test images: 300

✓ Wrote 300 predictions to /content/work/predictions.json
✓ Best model saved to /content/work/best_model.pt

Sample predictions:
  Egypt-New Capital City/New_Capital_Phase3-Part-2/815/DJI_20250325112358_0006.JPG -> {'mark': {'x': 3674.99, 'y': 260.68}, 'verified_shape': 'Square'}
  Egypt-New Capital City/New_Capital_Phase3-Part-2/g207p770/DJI_20250319115738_0079.JPG -> {'mark': {'x': 1967.27, 'y': 1346.48}, 'verified_shape': 'Square'}
  Egypt-New Capital City/New_Capital_Phase3-Part-2/T174/DJI_20250323114702_0159.JPG -> {'mark': {'x': 1498.67, 'y': 1481.45}, 'verified_shape': 'Square'}


In [26]:
from collections import Counter
shape_dist = Counter(p['verified_shape'] for p in predictions.values())
print("Predicted shape distribution on test set:")
for k, v in shape_dist.items():
    print(f"  {k}: {v} ({100*v/300:.1f}%)")


Predicted shape distribution on test set:
  Square: 86 (28.7%)
  L-Shape: 69 (23.0%)
  Cross: 145 (48.3%)


In [27]:
# Confirm JSON structure matches the spec exactly
import json
with open(os.path.join(BASE, 'predictions.json')) as f:
    p = json.load(f)

# Pick any 5 to inspect
sample = list(p.items())[:5]
for k, v in sample:
    print(k)
    print(f"  -> {v}")
    # Verify structure
    assert 'mark' in v and 'verified_shape' in v
    assert 'x' in v['mark'] and 'y' in v['mark']
    assert v['verified_shape'] in ['Cross', 'Square', 'L-Shape']
print(f"\nTotal entries: {len(p)}")
print("Structure: ✓ all entries valid")


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/GCP_Assignment_Datasets/predictions.json'

In [28]:
import os, json

local_path = '/content/work/predictions.json'
drive_path = os.path.join(BASE, 'predictions.json')

print("Local file exists:", os.path.exists(local_path))
if os.path.exists(local_path):
    print(f"  Size: {os.path.getsize(local_path)} bytes")

print("Drive file exists:", os.path.exists(drive_path))

print("\nContents of BASE:")
for item in sorted(os.listdir(BASE)):
    full = os.path.join(BASE, item)
    if os.path.isfile(full):
        print(f"  FILE  {item}  ({os.path.getsize(full)} bytes)")
    else:
        print(f"  DIR   {item}")

print(f"\nBASE = {BASE}")


Local file exists: True
  Size: 50883 bytes
Drive file exists: False

Contents of BASE:
  DIR   test_dataset
  DIR   train_dataset

BASE = /content/drive/MyDrive/GCP_Assignment_Datasets


In [29]:
import shutil, os, time

src_pred  = '/content/work/predictions.json'
src_model = '/content/work/best_model.pt'
dst_pred  = os.path.join(BASE, 'predictions.json')
dst_model = os.path.join(BASE, 'best_model.pt')

# Copy with overwrite
shutil.copy(src_pred, dst_pred)
shutil.copy(src_model, dst_model)
print("Copied. Forcing Drive flush...")

# Force Drive to sync
from google.colab import drive
drive.flush_and_unmount()
time.sleep(3)
drive.mount('/content/drive')

# Verify
print("\nAfter flush & remount:")
print(f"  predictions.json exists: {os.path.exists(dst_pred)}")
if os.path.exists(dst_pred):
    print(f"  predictions.json size:   {os.path.getsize(dst_pred)} bytes")
print(f"  best_model.pt exists:    {os.path.exists(dst_model)}")
if os.path.exists(dst_model):
    print(f"  best_model.pt size:      {os.path.getsize(dst_model)/1e6:.1f} MB")


OSError: [Errno 30] Read-only file system: '/content/drive/MyDrive/GCP_Assignment_Datasets/predictions.json'

In [30]:
from google.colab import files
files.download('/content/work/predictions.json')
files.download('/content/work/best_model.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [31]:
import os, shutil
my_drive = '/content/drive/MyDrive'
out_dir = os.path.join(my_drive, 'gcp_submission')
os.makedirs(out_dir, exist_ok=True)
shutil.copy('/content/work/predictions.json', os.path.join(out_dir, 'predictions.json'))
shutil.copy('/content/work/best_model.pt', os.path.join(out_dir, 'best_model.pt'))
print("Saved to:", out_dir)
print(os.listdir(out_dir))


Saved to: /content/drive/MyDrive/gcp_submission
['predictions.json', 'best_model.pt']
